# Data Challenge : Lynred data

---

## Imports

In [28]:
# --- Standard Library ---
import os
import glob
import csv
import json

# --- Math & Image Processing ---
import numpy as np
import cv2
from skimage import io
from scipy.ndimage import convolve1d, gaussian_filter1d
from scipy.signal import find_peaks

# --- Parallelization ---
from joblib import Parallel, delayed

# --- Visualization ---
import matplotlib.pyplot as plt

# --- Machine Learning ---
import xgboost as xgb
from scipy.ndimage import uniform_filter1d

from collections import defaultdict
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
from pathlib import Path

---

## Load Image

In [14]:
def load_img(path):
    """
    Loads an image from the given file path.

    Args:
        path (str): Full path to the image file.

    Returns:
        np.ndarray: The image as a numpy array.
    """
    # Simply read and return the image
    return io.imread(path)

---

## Build all the data set paths

In [15]:
def load_dataset(folder='train', high_dyn=True):
    """
    Builds a dictionary of image paths grouped by type, sequence, and dynamics.

    Args:
        folder (str): Target directory name.
        high_dyn (bool): Whether to include 'high_dyn' in the search.

    Returns:
        tuple: (Dataset dictionary, Flat list of all image paths)
    """
    
    cam_types = ['HD', 'SXGA', 'VGA']
    seqs = ['sequence_1', 'sequence_2', 'sequence_3']
    
    # Define dynamics based on the high_dyn flag
    dyns = [
        'low dyn', 
        'low dyn with columns 1', 
        'low dyn with columns 2', 
        'low dyn with columns 3'
    ]
    if high_dyn:
        dyns.insert(0, 'high_dyn')

    data_dict = {}
    all_paths = []

    # Build the dictionary and flat list
    for t in cam_types:
        data_dict[t] = {}
        for seq in seqs:
            data_dict[t][seq] = {}
            for dyn in dyns:
                # Grab all PNG files in the specific folder
                pattern = f"{folder}/{t}/{seq}/{dyn}/*.png"
                paths = glob.glob(pattern)
                
                data_dict[t][seq][dyn] = paths
                all_paths.extend(paths)

    return data_dict, all_paths

--- 

## Correction

In [16]:
def make_mask(shape, defects):
    """
    Creates a 2D boolean mask for defective pixels based on coordinates.

    Args:
        shape (tuple): The (height, width) of the target image.
        defects (dict): Keys are x-coords, values are dicts with 'start' and 'stop' y-coords.
            
    Returns:
        np.ndarray: 2D boolean array (True = defect).
    """
    h, w = shape
    mask = np.zeros(shape, dtype=bool)
    
    # Return empty mask if no defects are provided
    if not defects:
        return mask

    for x, intervals in defects.items():
        # Security : Skip if x-coordinate is out of image bounds
        if not (0 <= x < w):
            continue

        # Apply True to the mask for each vertical interval on this column
        for y0, y1 in intervals:
            y_start = max(0, int(y0))
            y_stop = min(h, int(y1))
            mask[y_start:y_stop, x] = True
            
            
    return mask

In [17]:
def fix_stripes(img, defects):
    """
    Corrects column defects using frequency separation and 1D interpolation.
    Non-defective pixels are preserved completely.

    Args:
        img (np.ndarray): 2D input image.
        defects (dict): Defect coordinates dictionary.
            
    Returns:
        np.ndarray: Corrected image
    """
    out_img = np.copy(img)
    h, w = out_img.shape
    
    # Generate the exact 2D mask
    mask = make_mask(out_img.shape, defects)
                    
    if not np.any(mask):
        return img.astype(np.uint16)

    img_float = img.astype(np.float32)

    # Vertical Frequency Separation
    low_freq = gaussian_filter1d(img_float, sigma=11.0, axis=0)
    high_freq = img_float - low_freq 

    low_fixed = np.copy(low_freq)
    
    # Horizontal Interpolation on Low Frequencies
    for y in range(h):
        row_mask = mask[y, :]
        if not np.any(row_mask):
            continue
            
        clean_idx = np.where(~row_mask)[0]
        err_idx = np.where(row_mask)[0]
        
        # Interpolate missing low-frequency pixels using clean neighbors
        if len(clean_idx) > 1:
            interp_vals = np.interp(err_idx, clean_idx, low_freq[y, clean_idx])
            low_fixed[y, err_idx] = interp_vals

    # Recombine and format output
    final_float = low_fixed + high_freq
    fixed_16b = np.clip(np.round(final_float), 0, 65535).astype(np.uint16)
    
    # Apply corrections only to defective areas
    out_img[mask] = fixed_16b[mask]
    
    return out_img.astype(np.uint16)

--- 

## Detection

In [18]:
def extract_image_features_vectorized(img_prev, img_curr, img_next):
    # Conversion en float64 pour toute l'image d'un coup
    img_prev = img_prev.astype(np.float64)
    img_curr = img_curr.astype(np.float64)
    img_next = img_next.astype(np.float64)
    
    height, width = img_curr.shape[:2]
    
    # DÉTECTION DE LA RÉSOLUTION (Pour la fenêtre du LT_moy)
    if height < 700: 
        taille_fenetre = 18
    elif height < 1050: 
        taille_fenetre = 31
    else: 
        taille_fenetre = 24

    # --- Statistiques Globales ---
    col_mean = np.mean(img_curr, axis=0) # Vecteur avec la moyenne de chaque colonne
    col_energy = np.sum(img_curr ** 2, axis=0) / height
    
    # =================================================
    # LT_moy (Fenêtre glissante )
    
    # uniform_filter1d fait la tendance locale pour TOUTES les colonnes instantanément
    tendance_locale = uniform_filter1d(col_mean, size=taille_fenetre, mode='reflect')
    
    # Calcul de l'écart-type local via la variance : V(X) = E(X^2) - E(X)^2
    mean_sq = np.mean(img_curr ** 2, axis=0)
    tendance_sq = uniform_filter1d(mean_sq, size=taille_fenetre, mode='reflect')
    std_locale = np.sqrt(np.maximum(tendance_sq - tendance_locale**2, 0))
    
    ecart_lt_moy = np.abs(col_mean - tendance_locale)
    ratio_lt_moy = ecart_lt_moy / (std_locale + 1e-5)
    
    # =========================================================================
    # Spatial & Temporel
    
    # Décalage du vecteur pour comparer avec la colonne de gauche et de droite
    left_neighbor = np.roll(col_mean, 1)
    right_neighbor = np.roll(col_mean, -1)
    neighbor_mean = (left_neighbor + right_neighbor) / 2.0
    
    # Correction des extrêmes (bords de l'image)
    neighbor_mean[0] = col_mean[1]
    neighbor_mean[-1] = col_mean[-2]
    
    spatial_diff = np.abs(col_mean - neighbor_mean)
    
    # Contexte temporel
    mean_prev = np.mean(img_prev, axis=0)
    mean_next = np.mean(img_next, axis=0)
    diff_temp_absolue = np.abs(col_mean - mean_prev)
    scintillement_temporel = np.abs(col_mean - ((mean_prev + mean_next) / 2.0))

    # Assemblage Final
    features_matrix = np.column_stack((
        ecart_lt_moy, ratio_lt_moy,
        spatial_diff, diff_temp_absolue, scintillement_temporel,
        col_mean, col_energy
    ))
    
    return features_matrix

In [19]:
def clf_detect(img_prev, img_curr, img_next, camera_type='VGA'):
    """
    Predict if a column is defect or not using the saved XGBoost baseline.

    Returns the list of defective column indices (x).
    """
    # Loading the model
    clf = xgb.XGBClassifier()
    # We load the model that we fit earlier with a specific camera_type
    chemin_modele = f'models/xgboost_{camera_type.lower()}_baseline.json'
    clf.load_model(chemin_modele)
    
    # Extract the same features that we use earlier 
    X_features = extract_image_features_vectorized(img_prev, img_curr, img_next)
    
    # We predict the columns type (sane or defects)
    predictions = clf.predict(X_features)
    
    # Index of defects columns
    colonnes_defectueuses = np.where(predictions == 1)[0]
    
    # We return the list to detect in these columns the fragmented one and doing a specific correction
    return colonnes_defectueuses.tolist()

### Version alternative avec local threshold en attendant la version avec le random forest ou xgboost

In [20]:
def local_threshold(image, window_size=50, std_factor=1.0):
    metric_ = np.mean(image, axis=0)
    
    smoothing_filter = np.ones(window_size) / window_size
    local_trend = convolve1d(metric_, smoothing_filter, mode='reflect')
    tolerance_margin = std_factor * np.std(metric_)
    
    defect_columns = np.where(np.abs(metric_ - local_trend) > tolerance_margin)[0]

    return [int(col) for col in defect_columns]

In [21]:
def merge_segments(extended_segments, tolerance=15):
    """
    Merges overlapping or closely spaced vertical segments within the same column.

    Args:
        extended_segments (list): A list of segments in the format [x, y_start, y_end].
        tolerance (int, optional): The maximum vertical gap (in pixels) allowed between 
            two segments for them to be merged. Defaults to 15.

    Returns:
        dict: A dictionary mapping column indices to a list of merged segment 
            intervals, formatted as {x: [(y_start, y_end), ...]}.
    """
    if not extended_segments:
        return {}

    intermediate_dict = defaultdict(list)
    for segment in extended_segments:
        x = segment[0]
        y_start = segment[1]
        y_end = segment[2]
        intermediate_dict[x].append((y_start, y_end))

    final_dict = {}
    for x, segment_list in intermediate_dict.items():
        if len(segment_list) <= 1:
            final_dict[x] = segment_list
            continue
            
        sorted_segments = sorted(segment_list, key=lambda coord: coord[0])
        merged_segments = [sorted_segments[0]]
        
        for current_segment in sorted_segments[1:]:
            last_valid_segment = merged_segments[-1]
            current_start, current_end = current_segment
            last_start, last_end = last_valid_segment
            
            if current_start <= (last_end + tolerance):
                new_end = max(last_end, current_end)
                merged_segments[-1] = (last_start, new_end)
            else:
                merged_segments.append(current_segment)
                
        final_dict[x] = merged_segments

    return final_dict


def detect_defects(img, target_columns_array, rupture_multiplier=3.5, smoothing_size=11):
    """
    Detects and isolates vertical defects in specified columns using a 1D region-growing algorithm.

    The algorithm identifies anomaly peaks (seeds) in a smoothed column signal and expands 
    them vertically until a sharp gradient (rupture) is encountered. A safety margin is 
    applied to the resulting segments before merging.

    Args:
        img (numpy.ndarray): The 2D input image array.
        target_columns_array (array-like): Indices of the columns to analyze.
        rupture_multiplier (float, optional): Multiplier for the standard deviation 
            of local differences to define stopping boundaries (walls). Defaults to 3.5.
        smoothing_size (int, optional): Window size for the 1D smoothing filter applied 
            to the columns. Defaults to 11.

    Returns:
        dict: A dictionary of detected and merged defect segments, mapped by column index.
    """
    height = img.shape[0]
    extended_segments = []
    
    smoothing_filter = np.ones(smoothing_size) / smoothing_size
    target_columns = np.atleast_1d(target_columns_array)

    for x in target_columns:
        x = int(x)
        column = img[:, x].astype(np.float32)
        
        smoothed_column = convolve1d(column, smoothing_filter, mode='reflect')
        col_median = np.median(smoothed_column)
        anomaly_signal = np.abs(smoothed_column - col_median)
        
        noise_threshold = 2 * np.std(anomaly_signal) 
        seeds, _ = find_peaks(anomaly_signal, height=noise_threshold, distance=10)
        
        if len(seeds) == 0:
            seeds = [np.argmax(anomaly_signal)]

        diffs = np.abs(np.diff(column))
        normal_noise = np.median(diffs)
        step_std = np.std(diffs)
        break_threshold = normal_noise + (rupture_multiplier * step_std)

        walls = np.where(diffs >= break_threshold)[0]

        for y_seed in seeds:
            upper_walls = walls[walls < y_seed]
            y_start = int(upper_walls[-1] + 1) if len(upper_walls) > 0 else 0

            lower_walls = walls[walls >= y_seed]
            y_end = int(lower_walls[0]) if len(lower_walls) > 0 else height - 1
            
            margin = 15  # You can test 10, 15, or 20 here!
            y_start = max(0, y_start - margin)
            y_end = min(height - 1, y_end + margin)
                
            extended_segments.append([x, y_start, y_end])

    return merge_segments(extended_segments)

---

## Run

In [22]:
# --- Configuration by Image Type ---

# Optimized parameters per type with Optuna 
PARAMS_LT = {
     'VGA':  {'window_size': 16, 'std_factor': 1.918},
     'HD':   {'window_size': 24, 'std_factor': 1.26},
     'SXGA': {'window_size': 31, 'std_factor': 1.74}
}

PARAMS_REGION_GROWING = { 
 'VGA': {'rupture_multiplier': 5.9,  'smoothing_size': 21},  
 'SXGA': {'rupture_multiplier': 6,   'smoothing_size': 15},  
 'HD':  {'rupture_multiplier': 3.84, 'smoothing_size': 3} 
}

def process_img(cam_type, seq, dyn, img_path):
    """
    Processes a single image: loads, detects defects, fixes them, and saves the result.

    Args:
        cam_type (str): 'VGA', 'SXGA', or 'HD'.
        seq (str): Sequence name.
        dyn (str): Dynamics type.
        img_path (str): Full path to the input image.
    """
    filename = os.path.basename(img_path)
    folder = os.path.dirname(img_path)
    
    res_folder = os.path.join(folder, "results")
    os.makedirs(res_folder, exist_ok=True)
    save_path = os.path.join(res_folder, filename)

    img = load_img(img_path)
    
    # Default values if missing
    p_rg = PARAMS_REGION_GROWING.get(cam_type, {'rupture_multiplier': 3.5, 'smoothing_size': 11}) 
    p_seed = PARAMS_LT.get(cam_type, {'window_size': 50, 'std_factor': 1.0})

    # Defect detection
    suspect_columns = local_threshold(img, **p_seed)
    
    defects = detect_defects(
        img, 
        target_columns_array=suspect_columns,
        rupture_multiplier=p_rg['rupture_multiplier'], 
        smoothing_size=p_rg['smoothing_size']                  
    )
    
    # Image correction if defects found
    if not defects:
        fixed_img = np.copy(img)
    else:
        fixed_img = fix_stripes(img, defects)

    # Output save
    out_img = np.clip(fixed_img, 0, 65535).astype(np.uint16) 
    
    cv2.imwrite(save_path, out_img)
    success = cv2.imwrite(save_path, out_img)
    
    if not success:
        print(f"ERROR: OpenCV failed to save {save_path}")
        return None

In [23]:
# --- MAIN RUN SCRIPT ---
data_dict, _ = load_dataset(folder='train', high_dyn=False)
tasks = []
for t, seqs in data_dict.items():
    for seq, dyns in seqs.items():
        for dyn, paths in dyns.items():
            if dyn == "low dyn":
                continue 
            for path in paths:
                tasks.append((t, seq, dyn, path))

print(f" Launching Joblib for {len(tasks)} images...")

# pour Parallel Execution
raw_results = Parallel(n_jobs=-1)(
    delayed(process_img)(*task) for task in tasks
)

print("GOOD : Joblib Over !") # indique la fin 

 Launching Joblib for 0 images...
GOOD : Joblib Over !


---

## Evaluation

In [24]:
# Evaluate sequence (from metrics.py)

def read_json(p):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(p)
    return json.loads(p.read_text(encoding="utf‑8"))

def process_json(data):
    # Returns a boolean mask (True if defective column)
    indices = []
    for c in data:
        indices += c["x_coord"]
    return np.array(indices, dtype=int)

def load_pair(gt_path, def_path, res_path):
    """Lit deux images en float32 (sans copie supplémentaire)."""
    # -1 = IMREAD_UNCHANGED for 16bits images
    gt  = cv2.imread(gt_path, -1).astype(np.float32, copy=False)
    default = cv2.imread(def_path, -1).astype(np.float32, copy=False)
    res = cv2.imread(res_path, -1).astype(np.float32, copy=False)
    return gt, default, res

def evaluate_sequence(args):
    sensor, seq_path, simulation_idx = args
    gt_path  = os.path.join(seq_path, "low dyn")
    def_path = os.path.join(seq_path,
                             f"low dyn with columns {simulation_idx}")
    res_path = os.path.join(def_path, "result")

    # Verification
    if not (os.path.isdir(gt_path) and os.path.isdir(def_path) and os.path.isdir(res_path)):
        return None

    # JSON for defective columns information
    json_file = next((f for f in os.listdir(def_path) if f.lower().endswith(".json")), None)
    if json_file is None:
        return None
    true_def = process_json(read_json(os.path.join(def_path, json_file)))
    nb_cols   = cv2.imread(sorted([os.path.join(gt_path, f) for f in os.listdir(gt_path) if f.lower().endswith(".png")])[0], -1).shape[1]
    true_mask = np.isin(np.arange(nb_cols), true_def)          # booléen 1‑D

    # List of files
    gt_files = sorted([os.path.join(gt_path, f) for f in os.listdir(gt_path) if f.lower().endswith(".png")])
    def_files = sorted([os.path.join(def_path, f) for f in os.listdir(gt_path) if f.lower().endswith(".png")])
    res_files = sorted([os.path.join(res_path, f) for f in os.listdir(res_path) if f.lower().endswith(".png")])
    if len(gt_files) != len(res_files):
        return None

    # Agregates
    tp, fp, fn = 0, 0, 0
    sq_def, cnt_def = 0.0, 0
    sq_ok , cnt_ok  = 0.0, 0

    # Parallel reading
    with ThreadPoolExecutor(max_workers=4) as pool:
        for gt_f, def_f, res_f in zip(gt_files, def_files, res_files):
            gt, default, res = pool.submit(load_pair, gt_f, def_f, res_f).result()

            # Residuals by column
            residu_detect = ((res - default) ** 2).sum(axis=0)
            detected = np.flatnonzero(residu_detect)

            # TP / FP / FN
            tp += np.intersect1d(detected, true_def, assume_unique=True).size
            fp += np.setdiff1d(detected, true_def, assume_unique=True).size
            fn += np.setdiff1d(true_def, detected, assume_unique=True).size

            residu_sq = ((res - gt) ** 2).sum(axis=0)
            # Separated RMSE
            mask_def = true_mask
            mask_ok  = ~true_mask

            sq_def += (residu_sq[mask_def]).sum()
            cnt_def += mask_def.sum() * gt.shape[0]

            sq_ok  += (residu_sq[mask_ok]).sum()
            cnt_ok += mask_ok.sum() * gt.shape[0]

    # Normalization by number of columns
    nb_cols_f = float(nb_cols * len(gt_files))
    tp /= nb_cols_f; fp /= nb_cols_f; fn /= nb_cols_f

    # Metrics
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    rmse_def = np.sqrt(sq_def / cnt_def) if cnt_def else 0.0
    rmse_ok  = np.sqrt(sq_ok  / cnt_ok ) if cnt_ok  else 0.0

    rmse_def_norm = 1.0 - min([rmse_def / 40, 1.0])
    rmse_ok_norm = 1.0 - min([rmse_ok / 40, 1.0])
    overall = (0.34 * f1 +
               0.33 * (rmse_def_norm) +
               0.33 * (rmse_ok_norm))

    # Tuple results
    return (sensor,
            os.path.basename(seq_path),
            def_path,
            f"{tp:.6f}",
            f"{fp:.6f}",
            f"{fn:.6f}",
            f"{prec:.6f}",
            f"{rec:.6f}",
            f"{f1:.6f}",
            f"{rmse_def:.6f}",
            f"{rmse_ok:.6f}",
            f"{rmse_def_norm:.6f}",
            f"{rmse_ok_norm:.6f}",
            f"{overall:.6f}",
            f"[{sensor}/{os.path.basename(seq_path)} {simulation_idx}] "
            f"TP={tp:.3f} FP={fp:.3f} FN={fn:.3f} "
            f"Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f} "
            f"RMSE_def={rmse_def:.2f} RMSE_ok={rmse_ok:.2f} "
            f"RMSE_def_norm={rmse_def_norm:.2f} RMSE_ok_norm={rmse_ok_norm:.2f} "
            f"Score final : {overall:.4f}")


In [25]:
# --- Configuration ---

root_path = "train" 
sensors   = ["HD", "SXGA", "VGA"]
csv_path  = "results.csv"

# construction tasks 
tasks = []
for sensor in sensors:
    sensor_path = os.path.join(root_path, sensor)
    if os.path.exists(sensor_path):
        for entry in os.scandir(sensor_path):
            if entry.is_dir():
                for sim in (1, 2, 3):
                    tasks.append((sensor, entry.path, sim))

print(f"Starting evaluation for {len(tasks)} tasks...\n" + "-"*60)

results = []
final_scores = []

# pour execution parallèle 
with ProcessPoolExecutor() as pool:
    # Submit all tasks to the pool
    futures = {pool.submit(evaluate_sequence, task): task for task in tasks}    
    for future in as_completed(futures):
        res = future.result()
        if res is not None:
            results.append(res)
            final_scores.append(float(res[13]))
            print(res[-1])

print("-" * 60)

# CSV Writing (que une fois que fini) 
if results:
    with open(csv_path, mode="w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([
            "sensor", "sequence", "def_path", "TP", "FP", "FN",
            "precision", "recall", "F1",
            "RMSE_def", "RMSE_ok", "RMSE_def_norm",
            "RMSE_ok_norm", "final_score"
        ])
        for r in results:
            writer.writerow(r[:-1])

# Final Score 
if final_scores:
    mean_score = np.mean(final_scores)
    print("\nEvaluation completed! CSV file generated.")
    print("=== MEAN FINAL SCORE ACROSS ALL SEQUENCES ===")
    print(f"Mean score = {mean_score:.6f}")
else:
    print("\nNo results were produced.") # si y'a une erreur 

Starting evaluation for 0 tasks...
------------------------------------------------------------
------------------------------------------------------------

No results were produced.


# To produce corrected images

In [33]:
# --- CUSTOM INFERENCE SCRIPT ---
import os
from joblib import Parallel, delayed

# 1. SET YOUR PATHS HERE
input_folder = r"C:\Users\Shivraj Sarode\Desktop\DC_Pics\train\SXGA\sequence_3\low dyn with columns 3"  # Replace with your actual input path
output_folder = os.path.join(input_folder, "corrected_output_vga")

# 2. DEFINE THE SENSOR TYPE 
# This is used to pick the right ML model and threshold parameters
# Options: 'VGA', 'HD', or 'SXGA'
camera_type = 'SXGA' 

# 3. PREPARE THE TASKS
if not os.path.exists(input_folder):
    print(f"ERROR: The path {input_folder} does not exist.")
else:
    os.makedirs(output_folder, exist_ok=True)
    
    # We find all .png images in the folder
    image_paths = [os.path.join(input_folder, f) for f in os.listdir(input_folder) 
                   if f.lower().endswith('.png')]
    
    print(f"Found {len(image_paths)} images. Starting correction...")

    # We reuse the process_img function from your notebook.
    # Note: process_img internaly creates a 'results' folder inside the input path.
    # To keep your notebook logic exactly as is, we call it directly.
    
    # Parallel execution to speed up the process
    Parallel(n_jobs=-1)(
        delayed(process_img)(
            cam_type=camera_type, 
            seq="inference",      # Placeholder name
            dyn="custom",         # Placeholder name
            img_path=path
        ) for path in image_paths
    )

    print(f"Done! Corrected images are located in: {os.path.join(input_folder, 'results')}")

Found 500 images. Starting correction...
Done! Corrected images are located in: C:\Users\Shivraj Sarode\Desktop\DC_Pics\train\SXGA\sequence_3\low dyn with columns 3\results


# To play corrected images like a movie

In [ ]:
# To play corrected images like a movie
import cv2
import os
import time

def play_corrected_movie(output_path, img_type, seq_no, fps=24):
    """
    Displays the corrected images from a specific folder as a movie.
    
    Args:
        output_path (str): Path to the folder containing corrected .png images.
        img_type (str): Format name (e.g., 'HD', 'VGA') for display.
        seq_no (str): Sequence name (e.g., 'sequence_3') for display.
        fps (int): Frames per second.
    """
    # 1. Get and sort all image files
    files = sorted([f for f in os.listdir(output_path) if f.endswith('.png')])
    
    if not files:
        print(f"No images found in {output_path}")
        return

    # Calculate wait time in milliseconds (1000ms / fps)
    delay = int(1000 / fps)
    
    window_name = f"Correction Preview: {img_type} - {seq_no}"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    print(f"Playing at {fps} FPS. Press 'q' to stop.")

    for fname in files:
        img_path = os.path.join(output_path, fname)
        # Read as 16-bit
        img = cv2.imread(img_path, -1)
        
        if img is None:
            continue

        # 2. Normalize for display (convert 16-bit to 8-bit for the screen)
        # We use min-max scaling so the video isn't too dark to see
        disp_img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        
        # 3. Add Text Overlay
        info_text = f"{img_type} | {seq_no} | Frame: {fname}"
        cv2.putText(disp_img, info_text, (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        # 4. Display
        cv2.imshow(window_name, disp_img)

        # 5. Timing logic
        # waitKey(delay) waits exactly long enough to maintain the FPS
        if cv2.waitKey(delay) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()
    print("Movie playback finished.")


# EXAMPLE Execution
play_corrected_movie(r"C:\Users\Shivraj Sarode\Desktop\DC_Pics\train\VGA\sequence_1\low dyn with columns 1\results", 'VGA', 'sequence_1', fps=24)

# For side by side comparision and saving

In [35]:
# --- SIDE-BY-SIDE PLAYER AND VIDEO SAVER (CORRECTED) ---
import cv2
import os
import numpy as np

def play_and_save_comparison(input_folder, output_video_name="comparison_result.mp4", fps=24, save_video=True):
    """
    Plays original and corrected images side-by-side and saves the result as an MP4.
    """
    # 1. Setup Paths
    results_folder = os.path.join(input_folder, "results")
    if not os.path.exists(results_folder):
        print(f"ERROR: Results folder not found at {results_folder}")
        return

    # Get and sort original files
    files = sorted([f for f in os.listdir(input_folder) if f.lower().endswith('.png')])
    if not files:
        print("No images found in the input folder.")
        return

    # 2. Initialize Video Writer
    first_img = cv2.imread(os.path.join(input_folder, files[0]), -1)
    h, w = first_img.shape
    combined_size = (w * 2, h)
    
    video_writer = None
    if save_video:
        # FIX: Changed cv2.fourcc to cv2.VideoWriter_fourcc
        fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
        video_writer = cv2.VideoWriter(output_video_name, fourcc, fps, combined_size)
        print(f"Saving video to: {os.path.abspath(output_video_name)}")

    window_name = "Side-by-Side Comparison (Left: Original | Right: Corrected)"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    delay = int(1000 / fps)

    try:
        for fname in files:
            img_orig = cv2.imread(os.path.join(input_folder, fname), -1)
            img_corr = cv2.imread(os.path.join(results_folder, fname), -1)

            if img_orig is None or img_corr is None:
                continue

            # Normalize for display/saving
            disp_orig = cv2.normalize(img_orig, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            disp_corr = cv2.normalize(img_corr, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

            # Concatenate horizontally
            combined_frame = np.hstack((disp_orig, disp_corr))
            combined_frame_bgr = cv2.cvtColor(combined_frame, cv2.COLOR_GRAY2BGR)

            # Add labels
            cv2.putText(combined_frame_bgr, "ORIGINAL", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            cv2.putText(combined_frame_bgr, "CORRECTED", (w + 50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow(window_name, combined_frame_bgr)
            
            if video_writer:
                video_writer.write(combined_frame_bgr)

            if cv2.waitKey(delay) & 0xFF == ord('q'):
                break
    finally:
        if video_writer:
            video_writer.release()
        cv2.destroyAllWindows()
        print("Processing finished.")

# --- EXECUTION ---
target_path = r"C:\Users\Shivraj Sarode\Desktop\DC_Pics\train\SXGA\sequence_3\low dyn with columns 3"
play_and_save_comparison(target_path, output_video_name="SXGA_Comparison.mp4", fps=24)

Saving video to: c:\Users\Shivraj Sarode\Desktop\SXGA_Comparison.mp4
Processing finished.
